In [ ]:
# 1. Imports
import os
import pandas as pd
import numpy as np
import joblib
import mlflow
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support
)


In [ ]:
# 2. Configurable parameters
config = {
    "run_name": "inference-eval",
    "model_path": "loan_risk_model.pth",  # should match the .pth saved in training
    "dataset_path": "/mnt/object/test.csv",  # CSV with same structure as training data
    "label_col": "risk_level",
    "experiment_name": "loan-risk-xgboost"  # same as training for continuity
}

In [ ]:
# 3. Load data
df = pd.read_csv(config["dataset_path"])
X = df.drop(columns=[config["label_col"]])
y = df[config["label_col"]]


In [ ]:
# 4. Load model
model = joblib.load(config["model_path"])


In [ ]:
# 5. Start MLflow run
mlflow.set_experiment(config["experiment_name"])

with mlflow.start_run(run_name=config["run_name"]):
    # Predict
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    # Accuracy
    acc = accuracy_score(y, y_pred)
    mlflow.log_metric("inference_accuracy", acc)

    # F1 Scores
    f1_macro = f1_score(y, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y, y_pred, average="weighted", zero_division=0)
    mlflow.log_metric("inference_f1_macro", f1_macro)
    mlflow.log_metric("inference_f1_weighted", f1_weighted)

    # Per-class metrics
    prec, rec, f1, support = precision_recall_fscore_support(y, y_pred, labels=[0, 1], zero_division=0)
    for cls_idx, cls_name in zip([0, 1], ["Low", "High"]):
        mlflow.log_metric(f"inference_precision_{cls_name}", prec[cls_idx])
        mlflow.log_metric(f"inference_recall_{cls_name}", rec[cls_idx])
        mlflow.log_metric(f"inference_f1_{cls_name}", f1[cls_idx])
        mlflow.log_metric(f"inference_support_{cls_name}", support[cls_idx])

    # Classwise confidence (based on prediction)
    confidence_tracker = {0: [], 1: []}
    for pred, prob in zip(y_pred, y_prob):
        confidence_tracker[pred].append(prob)
    for cls in [0, 1]:
        if confidence_tracker[cls]:
            avg_conf = np.mean(confidence_tracker[cls])
            mlflow.log_metric(f"inference_avg_confidence_class_{cls}", avg_conf)

    # ROC-AUC
    roc_auc = roc_auc_score(y, y_prob)
    mlflow.log_metric("inference_roc_auc", roc_auc)

    # Confusion Matrix
    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low", "High"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title("Inference Confusion Matrix")
    cm_path = "inference_confusion_matrix.png"
    plt.savefig(cm_path)
    plt.close()
    mlflow.log_artifact(cm_path)

    print(f"✅ Inference complete. Accuracy: {acc:.4f}, ROC-AUC: {roc_auc:.4f}")